In [1]:
#Task 2.1 

class Combatant: 
    def __init__(self,name,health,damage,cooldown): 
        self.name = name 
        self.health = health 
        self.damage = damage 
        self.cooldown = cooldown 

    def take_damage(self,points): 
        self.health -= points 

    def is_dead(self): 
        return self.health <= 0 

In [2]:
#Task 2.2 

class Slot: 
    def __init__(self,combatant): 
        self.combatant = combatant 
        self.turns_before_ready = self.combatant.cooldown 
        self.next_slot = None 

    def get_priority(self): 
        return self.turns_before_ready 

    def update_priority(self): 
        if self.turns_before_ready > 0: 
            self.turns_before_ready -= 1 

In [3]:
#Task 2.3 

class PriorityQueue: 
    def __init__(self): 
        self.head = None 
        self.count = 0 

    def add(self,combatant): 
        new_node = Slot(combatant) 

        if self.head == None or self.head.get_priority() > new_node.get_priority(): 
            new_node.next_slot = self.head 
            self.head = new_node
            self.count += 1 
            return 

        previous = None 
        current = self.head 

        while current and current.get_priority() <= new_node.turns_before_ready: 
            previous = current 
            current = current.next_slot 

        previous.next_slot = new_node 
        new_node.next_slot = current 
        self.count += 1 

    def remove(self,name): 
        if self.head == None: 
            return 

        if self.head.combatant.name == name: 
            self.head = self.head.next_slot 
            self.count -= 1 
            return 

        previous = None 
        current = self.head 

        while current and current.combatant.name != name: 
            previous = current 
            current = current.next_slot 

        if current == None: 
            return 

        previous.next_slot = current.next_slot 
        self.count -= 1 

    def dequeue_highest_priority(self): 
        if self.head == None or self.head.get_priority() > 0: 
            print("hi")
            return None 
        
        if self.head.turns_before_ready == 0:
            temp = self.head 

            self.head = self.head.next_slot 

            self.count -= 1  

            return temp.combatant 

    def update_slots(self): 
        current = self.head 

        while current: 
            current.update_priority()
            current = current.next_slot 

        print("Updating slots...")



    def get_alive_combatants(self):
        alive_combatants = []
        curr = self.head
        while curr is not None:
            alive_combatants.append(curr.combatant)
            curr = curr.next_slot
        return alive_combatants


In [4]:
#Task 2.4 

class Player(Combatant): 
    def __init__(self,name,health,damage,cooldown): 
        super().__init__(name,health,damage,cooldown) 

    def get_opponent_choice(self,lst_combatant): 
        for i in range(len(lst_combatant)): 
            print(f"{i+1}.{lst_combatant[i].name}")

        valid_index = [i for i in range(1,len(lst_combatant)+1)]

        valid_status = False

        while valid_status == False: 
            user_input = int(input("Choose an opponent"))

            if user_input in valid_index: 
                valid_status = True 

        return lst_combatant[user_input-1]


class Enemy(Combatant): 
    def __init__(self,name,health,damage,cooldown): 
        super().__init__(name,health,damage,cooldown) 

    def get_opponent_choice(self,lst_combatant): 
        for c in lst_combatant: 
            if isinstance(c,Player) == True: 
                return c  

with open("COMBATANTS.txt") as f: 
    combatants_info = [x.split(",") for x in f.read().split("\n")[1:-1]] 

lst_combatant = [] 

for combatant in combatants_info: 
    if combatant[0] == "player": 
        lst_combatant.append(Player(combatant[0],int(combatant[1]),int(combatant[2]),int(combatant[3])))
    else: 
        lst_combatant.append(Enemy(combatant[0],int(combatant[1]),int(combatant[2]),int(combatant[3])))

pq = PriorityQueue() 

print(combatants_info) 

for i in lst_combatant: 
    pq.add(i) 

[['player', '50', '8', '3'], ['mosquito', '5', '2', '2'], ['rat', '10', '5', '3'], ['crow', '20', '8', '5']]


In [5]:
#Task 2.5 

game_status = True 

while game_status == True: 

    print(pq.get_alive_combatants())
    while pq.dequeue_highest_priority() == None: 
        pq.update_slots()
        print(pq.get_alive_combatants())

    print(pq.get_alive_combatants())

    # print([i.get_priority() for i in pq.get_alive_combatants()])

    active_combatant = pq.dequeue_highest_priority()
    
    print(active_combatant)

    # pq.remove(active_combatant.name) 

    opponent = active_combatant.get_opponent_choice(lst_combatant) 

    opponent.take_damage(active_combatant.damage) 

    print(f"{active_combatant.name} dealt {active_combatant.damage} damage to {opponent.name}")

    if opponent.is_dead() == True: 
        if isinstance(opponent,Player) == True: 
            game_status = False
            win_status = False  

        print(f"{opponent.name} dies!")
        pq.remove(opponent.name)

    if pq.count == 1: 
        win_status = True 
        game_status = False 

if win_status == True: 
    print("You won!")
else: 
    print("You died!")






[<__main__.Enemy object at 0x0000026679577770>, <__main__.Player object at 0x0000026679577620>, <__main__.Enemy object at 0x00000266793BF750>, <__main__.Enemy object at 0x00000266793BF4D0>]
hi
Updating slots...
[<__main__.Enemy object at 0x0000026679577770>, <__main__.Player object at 0x0000026679577620>, <__main__.Enemy object at 0x00000266793BF750>, <__main__.Enemy object at 0x00000266793BF4D0>]
hi
Updating slots...
[<__main__.Enemy object at 0x0000026679577770>, <__main__.Player object at 0x0000026679577620>, <__main__.Enemy object at 0x00000266793BF750>, <__main__.Enemy object at 0x00000266793BF4D0>]
[<__main__.Player object at 0x0000026679577620>, <__main__.Enemy object at 0x00000266793BF750>, <__main__.Enemy object at 0x00000266793BF4D0>]
hi
None


AttributeError: 'NoneType' object has no attribute 'get_opponent_choice'